# YOLOv10n NSFW Detector

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.1 MB/s eta 0:00:00


In [ ]:
import os
import yaml
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import tensorflow as tf
from pathlib import Path
import shutil
import json
from datetime import datetime
import zipfile

In [ ]:
class ScreenContentDetector:
    def __init__(self, dataset_path, model_name='yolov10n'):
        """Initialize the screen content detector."""
        self.dataset_path = Path(dataset_path)
        self.model_name = model_name
        self.data_yaml = self.dataset_path / 'data.yaml'
        self.output_dir = Path('output')
        self.output_dir.mkdir(exist_ok=True)

    def prepare_dataset(self, classes=None):
      """Use existing data.yaml from the extracted dataset if available."""
      existing_yaml = self.dataset_path / 'data.yaml'

      if existing_yaml.exists():
          print(f"Using existing data.yaml found at: {existing_yaml}")
          self.data_yaml = existing_yaml
      else:
          # fallback: create a new one if missing
          print("No data.yaml found, creating a new one.")
          if classes is None:
              classes = ['safe', 'nudity', 'gore']

          data_config = {
              'path': str(self.dataset_path.absolute()),
              'train': 'train/images',
              'val': 'val/images',
              'names': {i: name for i, name in enumerate(classes)}
          }
          with open(self.data_yaml, 'w') as f:
              yaml.dump(data_config, f, default_flow_style=False)
          print(f"Created new data.yaml at: {self.data_yaml}")

      return self.data_yaml


    def train_model(self, epochs=50, imgsz=640, batch=16, weights='yolov10n.pt'):
        """Train YOLOv10n model. """

        print(f"Training model.")

        # Load pretrained YOLOv10n (COCO) weights for transfer learning
        model = YOLO(weights)

        # Select device from PyTorch (Colab GPU -> 0, else CPU)
        device = 0 if torch.cuda.is_available() else 'cpu'

        # Train the model
        results = model.train(
            data=str(self.data_yaml),
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=device,
            project='runs/train',
            name='screen_content_detector',
            patience=20,
            save=True,
            plots=True
        )

        # Get best model path
        self.best_model_path = Path('runs/train/screen_content_detector/weights/best.pt')
        print(f"Training complete! Best model saved at: {self.best_model_path}")

        return self.best_model_path

    def export_to_tflite(self, model_path=None):
        if model_path is None:
            model_path = self.best_model_path

        print(f"Export model")

        model = YOLO(model_path)

        # Export to quantized TFLite
        export_path = model.export(format='tflite', imgsz=640, half=True)

        # Add metadata if possible (optional but strongly recommended)
        try:
            from tflite_support import metadata as _metadata
            labelmap_path = self.output_dir / 'labelmap.txt'
            populator = _metadata.MetadataPopulator.with_model_file(str(export_path))
            populator.load_associated_files([str(labelmap_path)])
            populator.populate()
            print("Metadata attached successfully.")
        except Exception as e:
            print("Metadata not added:", e)

        tflite_output = self.output_dir / 'model.tflite'
        shutil.copy(export_path, tflite_output)
        print(f"TFLite model exported to: {tflite_output}")

        return tflite_output


    def create_label_file(self):
        """Create labelmap.txt."""
        # Read classes from data.yaml
        with open(self.data_yaml, 'r') as f:
            data = yaml.safe_load(f)

        classes = data['names']

        # Create labelmap.txt
        labelmap_path = self.output_dir / 'labelmap.txt'
        with open(labelmap_path, 'w') as f:
            for idx in sorted(classes.keys()):
                f.write(f"{classes[idx]}\n")

        print(f"Label map created at: {labelmap_path}")
        print(f"Classes: {list(classes.values())}")

        return labelmap_path

    def run_inference_samples(self, model_path=None, num_samples=5):
        """Run inference."""
        if model_path is None:
            model_path = self.best_model_path

        print(f"Run inference.")

        # Load model
        model = YOLO(model_path)

        # Get sample images from validation set
        val_images_dir = self.dataset_path / 'val' / 'images'
        sample_images = list(val_images_dir.glob('*.jpg'))[:num_samples]
        sample_images += list(val_images_dir.glob('*.png'))[:num_samples - len(sample_images)]

        if not sample_images:
            print("No sample images found in validation set")
            return

        # Create inference output directory
        inference_dir = self.output_dir / 'inference_results'
        inference_dir.mkdir(exist_ok=True)

        results_summary = []

        for idx, img_path in enumerate(sample_images[:num_samples]):
            print(f"\nProcessing sample {idx + 1}/{num_samples}: {img_path.name}")

            # Run inference
            results = model(str(img_path), conf=0.25)

            # Get predictions
            boxes = results[0].boxes
            img = cv2.imread(str(img_path))

            if len(boxes) > 0:
                for box in boxes:
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    class_name = model.names[cls_id]

                    # Get bounding box coordinates
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Set color based on class
                    color_map = {
                        'safe': (0, 255, 0),      # Green
                        'nudity': (0, 0, 255),    # Red
                        'gore': (255, 0, 0)       # Blue
                    }
                    color = color_map.get(class_name, (255, 255, 255))

                    # Draw bounding box
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

                    # Add label
                    label = f"{class_name}: {conf:.2f}"
                    cv2.putText(img, label, (x1, y1 - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                    results_summary.append({
                        'image': img_path.name,
                        'class': class_name,
                        'confidence': conf
                    })

                    print(f"  Detected: {class_name} (confidence: {conf:.3f})")
            else:
                print(f"  No detections")
                results_summary.append({
                    'image': img_path.name,
                    'class': 'none',
                    'confidence': 0.0
                })

            # Save annotated image
            output_path = inference_dir / f"result_{idx + 1}_{img_path.name}"
            cv2.imwrite(str(output_path), img)
            print(f"  Saved to: {output_path}")

        # Print summary
        print("INFERENCE SUMMARY")
        for result in results_summary:
            print(f"{result['image']:30s} -> {result['class']:10s} ({result['confidence']:.3f})")

        return results_summary

In [5]:
class TFLiteInference:
    """TFLite inference class for mobile deployment."""
    def __init__(self, model_path, label_path):
        """ Initialize TFLite interpreter."""
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=str(model_path))
        self.interpreter.allocate_tensors()

        # Get input and output details
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

        # Load labels
        with open(label_path, 'r') as f:
            self.labels = [line.strip() for line in f.readlines()]

        print(f" TFLite model loaded")
        print(f"  Input shape: {self.input_details[0]['shape']}")
        print(f"  Labels: {self.labels}")

    def preprocess_image(self, image_path, target_size=640):
        """Preprocess image for TFLite inference."""
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (target_size, target_size))
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        return img

    def infer(self, image_path):
        """Run inference on an image."""
        # Preprocess
        input_data = self.preprocess_image(image_path)

        # Run inference
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        self.interpreter.invoke()

        # Get output
        output_data = self.interpreter.get_tensor(self.output_details[0]['index'])

        return output_data

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from datetime import datetime
import json

class ModelEvaluator:
    """Evaluate model and save metrics"""
    def __init__(self, model_path, data_yaml, output_dir='output/metrics_results'):
        self.model_path = Path(model_path)
        self.data_yaml = Path(data_yaml)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Load YOLO model
        self.model = YOLO(str(self.model_path))

    def evaluate(self, imgsz=640, batch=16, conf=0.25):
        """Run validation and compute metrics."""
        print("Evaluate model")

        results = self.model.val(
            data=str(self.data_yaml),
            imgsz=imgsz,
            batch=batch,
            conf=conf,
            split='val'
        )

        # Extract metrics from the Ultralytics DetMetrics object (results.box)
        box = results.box
        try:
            f1 = float(np.mean(box.f1)) if getattr(box, "f1", None) is not None and len(box.f1) else 0.0
        except Exception:
            f1 = 0.0
        metrics = {
            "precision": float(box.mp),       # mean precision across classes
            "recall": float(box.mr),          # mean recall across classes
            "f1_score": f1,
            "map50": float(box.map50),
            "map50_95": float(box.map),
        }

        # Timestamp for uniqueness
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = self.output_dir / f"eval_{timestamp}"
        run_dir.mkdir(exist_ok=True)

        # Save results as JSON and TXT
        json_path = run_dir / "metrics_summary.json"
        txt_path = run_dir / "metrics_summary.txt"

        with open(json_path, 'w') as jf:
            json.dump(metrics, jf, indent=4)

        with open(txt_path, 'w') as tfile:
            for k, v in metrics.items():
                tfile.write(f"{k}: {v:.4f}\n")

        print(f" Evaluation complete. Results saved at: {run_dir}")
        print("\nMetrics Summary:")
        for k, v in metrics.items():
            print(f"  {k:12s}: {v:.4f}")

        return metrics


In [7]:
def verify_extraction(dataset_path):
    """Check if dataset extracted correctly and contains required files."""
    required_dirs = ['train/images', 'train/labels', 'val/images', 'val/labels']
    missing = [d for d in required_dirs if not Path(dataset_path, d).exists()]
    if missing:
        print("⚠️ Missing folders after extraction:")
        for m in missing:
            print(f"  - {m}")
        for m in missing:
            Path(dataset_path, m).mkdir(parents=True, exist_ok=True)
        print("Created missing directories.")
    else:
        print("✅ All required folders found.")

    import glob
    img_files = glob.glob(f"{dataset_path}/train/images/*.*")
    if not img_files:
        print("❌ No images found in train/images.")
    else:
        print(f"✅ Found {len(img_files)} training images.")

In [ ]:
def main():
    """Main training and export pipeline."""
    print("YOLOv10n")

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # --- Optional: load config.yaml if present (falls back to built-in defaults) ---
    cfg = {}
    if os.path.exists('config.yaml'):
        with open('config.yaml', 'r') as f:
            cfg = yaml.safe_load(f) or {}
        print("Loaded settings from config.yaml")
    else:
        print("config.yaml not found - using built-in defaults")

    ds_cfg    = cfg.get('dataset', {})
    train_cfg = cfg.get('train', {})
    eval_cfg  = cfg.get('eval', {})
    model_cfg = cfg.get('model', {})

    # Configuration
    DATASET_ZIP  = ds_cfg.get('zip_path', '/content/drive/MyDrive/Datasets/another_final_dataset.zip')
    DATASET_PATH = ds_cfg.get('extract_to', '/content/combined_dataset')
    CLASSES      = ds_cfg.get('classes', ['safe', 'nudity', 'gore'])
    WEIGHTS      = model_cfg.get('weights', 'yolov10n.pt')

    # Unzip dataset
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(DATASET_PATH)

    # --- Step 1: Initialize detector ---
    detector = ScreenContentDetector(
        dataset_path=DATASET_PATH,
        model_name='yolov10n'
    )

    # --- Step 2: Prepare dataset ---
    print("Prepare Dataset")
    detector.prepare_dataset(classes=CLASSES)

    # --- Step 3: Create label map (before export so TFLite metadata can attach) ---
    print("Create Label Map")
    labelmap_path = detector.create_label_file()

    # --- Step 4: Train model ---
    print("Train Model")
    model_path = detector.train_model(
        epochs=train_cfg.get('epochs', 50),
        imgsz=train_cfg.get('imgsz', 640),
        batch=train_cfg.get('batch', 16),
        weights=WEIGHTS,
    )

    # --- Step 5: Export to TFLite ---
    print("Export TFLite")
    tflite_path = detector.export_to_tflite(model_path)

    # --- Step 6: Run inference on samples ---
    print("Run Inference")
    detector.run_inference_samples(model_path, num_samples=5)

    # --- Step 7: Evaluate model ---
    print("Evaluate model")
    evaluator = ModelEvaluator(model_path, detector.data_yaml)
    metrics = evaluator.evaluate(imgsz=eval_cfg.get('imgsz', 640))

    # --- Step 8: Test TFLite inference ---
    print("Test TFLite inference")
    tflite_model = TFLiteInference(tflite_path, labelmap_path)

    print("Complete")
    print("Output files:")
    print(f"  - TFLite model: {tflite_path}")
    print(f"  - Label map: {labelmap_path}")
    print(f"  - Inference results: output/inference_results/")


In [ ]:
if __name__ == "__main__":
    main()

In [10]:
#import shutil

# Path to the folder you want to delete
#folder_path = 'sample_data'

# Deleting the folder
#shutil.rmtree(folder_path)

In [11]:
from google.colab import files
import shutil
import os
import zipfile
from tqdm.notebook import tqdm  # progress bar in Colab

def zip_with_progress(src_paths, zip_name="/content/final.zip"):
    file_paths = []
    for src in src_paths:
        if os.path.isdir(src):
            # Walk through directories
            for root, _, files in os.walk(src):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, os.path.commonpath(src_paths))
                    file_paths.append((full_path, arcname))
        elif os.path.isfile(src):
            # Single file
            arcname = os.path.basename(src)
            file_paths.append((src, arcname))

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for full_path, arcname in tqdm(file_paths, desc="Zipping files", unit="file"):
            zipf.write(full_path, arcname)

    return zip_name


# Example usage:
src_items = [
    "/content/output",             # folder
    "/content/runs",                # another folder
    "/content/yolo11n.pt",          # individual file
    "/content/calibration_image_sample_data_20x128x128x3_float32.npy"
]

zip_path = zip_with_progress(src_items, "/content/final.zip")

# Download the zip
files.download(zip_path)


Zipping files:   0%|          | 0/55 [00:00<?, ?file/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>